# Control Plane Bulk Writes with VARIANT Columns

This notebook demonstrates using FLEET-Q Control Plane for efficient bulk writes to Snowflake tables with VARIANT columns.

## Benefits:
- Automatic batching (pools operations for 15-30 seconds)
- ORM-agnostic grouping
- Dynamic writer scaling
- No manual connection management

## Prerequisites:
- FLEET-Q server running at http://localhost:8000
- Control plane enabled
- Snowflake credentials configured

## 1. Setup

In [ ]:
# Install required packages
!pip install httpx pandas

In [ ]:
import httpx
import asyncio
import json
import pandas as pd
import time
from typing import Dict, Any, List
from datetime import datetime

# FLEET-Q API endpoint
FLEETQ_URL = "http://localhost:8000"

print("✅ Imports successful")

## 2. Check Control Plane Status

In [ ]:
async def check_control_plane_status():
    """Verify control plane is running"""
    async with httpx.AsyncClient() as client:
        try:
            # Check root endpoint
            response = await client.get(f"{FLEETQ_URL}/")
            info = response.json()
            
            print("🚀 FLEET-Q Status:")
            print(f"   Service: {info['service']}")
            print(f"   Pod ID: {info['pod_id']}")
            print(f"   Control Plane: {'✅ Enabled' if info.get('control_plane_enabled') else '❌ Disabled'}")
            
            if not info.get('control_plane_enabled'):
                print("\n⚠️  Control plane is not enabled!")
                print("   Set FLEET_Q_ENABLE_CONTROL_PLANE=true and restart")
                return False
            
            # Get detailed stats
            response = await client.get(f"{FLEETQ_URL}/control-plane/stats")
            stats = response.json()
            
            print("\n📊 Control Plane Stats:")
            print(f"   Running: {stats['running']}")
            print(f"   Buffered Operations: {stats['buffer_stats']['total_buffered']}")
            print(f"   Active Workers: {stats['writer_pool']['active_workers']}")
            print(f"   Queue Depth: {stats['writer_pool']['queue_depth']}")
            
            return True
            
        except httpx.ConnectError:
            print("❌ Cannot connect to FLEET-Q")
            print("   Make sure server is running: uvicorn fleet_q.quickstart.main:app")
            return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False

# Check status
await check_control_plane_status()

## 3. Helper Functions for Bulk Writes

In [ ]:
async def submit_bulk_write(
    destination: str,
    data: Dict[str, Any],
    orm_type: str = "raw",
    priority: int = 0
) -> str:
    """Submit a write operation to control plane"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        response = await client.post(
            f"{FLEETQ_URL}/control-plane/write",
            json={
                "writer_type": "snowflake",
                "destination": destination,
                "data": data,
                "orm_type": orm_type,
                "priority": priority
            }
        )
        result = response.json()
        return result['operation_id']

async def get_control_plane_stats() -> Dict[str, Any]:
    """Get current control plane statistics"""
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{FLEETQ_URL}/control-plane/stats")
        return response.json()

async def trigger_flush() -> Dict[str, Any]:
    """Manually trigger flush of all buffers"""
    async with httpx.AsyncClient() as client:
        response = await client.post(f"{FLEETQ_URL}/control-plane/flush")
        return response.json()

def print_stats(stats: Dict[str, Any]):
    """Pretty print control plane stats"""
    print("\n" + "="*60)
    print("📊 CONTROL PLANE STATISTICS")
    print("="*60)
    
    buffer_stats = stats['buffer_stats']
    print(f"\n📦 Buffer:")
    print(f"   Total Buffered: {buffer_stats['total_buffered']}")
    print(f"   Buffer Groups: {buffer_stats['buffer_count']}")
    print(f"   Last Flush: {buffer_stats['last_flush_ago']:.1f}s ago")
    
    if buffer_stats['buffers']:
        print(f"\n   By Destination:")
        for key, count in buffer_stats['buffers'].items():
            print(f"      {key}: {count} ops")
    
    writer_pool = stats['writer_pool']
    print(f"\n⚙️  Writer Pool:")
    print(f"   Active Workers: {writer_pool['active_workers']}")
    print(f"   Queue Depth: {writer_pool['queue_depth']}")
    
    storage_stats = stats['storage_stats']
    print(f"\n💾 Storage:")
    print(f"   Pending: {storage_stats['pending_operations']}")
    print(f"   Completed: {storage_stats['completed_operations']}")
    print(f"   DB Size: {storage_stats['database_size_bytes'] / 1024:.2f} KB")
    print("="*60)

print("✅ Helper functions defined")

## 4. Example 1: Simple VARIANT Writes

In [ ]:
async def test_simple_variant_writes():
    """Submit simple writes with VARIANT columns"""
    print("📝 Example 1: Simple VARIANT Writes\n")
    
    # Prepare data with VARIANT columns
    # NOTE: Data dict should match your table structure
    events = [
        {
            'event_name': 'user_signup',
            'user_id': 1001,
            'status': 'success',
            # VARIANT column: event_data
            'event_data': json.dumps({
                'source': 'web',
                'browser': 'Chrome',
                'device': 'Desktop',
                'referrer': 'https://google.com'
            }),
            # VARIANT column: user_metadata
            'user_metadata': json.dumps({
                'name': 'Alice Johnson',
                'email': 'alice@example.com',
                'preferences': {
                    'newsletter': True,
                    'theme': 'dark'
                }
            })
        },
        {
            'event_name': 'product_view',
            'user_id': 1002,
            'status': 'success',
            'event_data': json.dumps({
                'product_id': 'PROD-123',
                'product_name': 'Widget Pro',
                'price': 99.99,
                'category': 'Electronics'
            }),
            'user_metadata': json.dumps({
                'name': 'Bob Smith',
                'email': 'bob@example.com',
                'loyalty_tier': 'gold'
            })
        }
    ]
    
    # Submit to control plane
    operation_ids = []
    for event in events:
        op_id = await submit_bulk_write(
            destination="VARIANT_TEST_TABLE",
            data=event,
            orm_type="raw"
        )
        operation_ids.append(op_id)
        print(f"✅ Submitted: {op_id}")
    
    # Check stats
    print("\n📊 Checking buffer status...")
    stats = await get_control_plane_stats()
    print_stats(stats)
    
    return operation_ids

# Run example
ops = await test_simple_variant_writes()

## 5. Example 2: Complex Nested VARIANT Data

In [ ]:
async def test_complex_variant_writes():
    """Submit writes with deeply nested VARIANT data"""
    print("📝 Example 2: Complex Nested VARIANT Data\n")
    
    # Complex nested structure
    complex_data = {
        'event_name': 'api_request',
        'user_id': 2001,
        'status': 'success',
        'event_data': json.dumps({
            'request': {
                'method': 'POST',
                'endpoint': '/api/v1/users/create',
                'headers': {
                    'Content-Type': 'application/json',
                    'Authorization': 'Bearer token123',
                    'User-Agent': 'Python/3.11'
                },
                'body': {
                    'user': {
                        'name': 'Test User',
                        'email': 'test@example.com',
                        'roles': ['admin', 'user', 'moderator'],
                        'settings': {
                            'notifications': {
                                'email': True,
                                'push': False,
                                'frequency': 'daily'
                            }
                        }
                    }
                }
            },
            'response': {
                'status_code': 201,
                'latency_ms': 145,
                'body': {
                    'success': True,
                    'user_id': 12345
                }
            },
            'metrics': {
                'db_queries': 3,
                'cache_hits': 5,
                'execution_time': 0.145
            }
        }),
        'user_metadata': json.dumps({
            'session_id': 'sess_abc123',
            'client_info': {
                'platform': 'web',
                'version': '2.1.0',
                'features': ['feature_a', 'feature_b', 'feature_c']
            }
        }),
        'raw_payload': json.dumps({
            'timestamp': datetime.now().isoformat(),
            'trace_id': 'trace_xyz789',
            'context': {
                'environment': 'production',
                'region': 'us-west-2',
                'availability_zone': 'us-west-2a'
            }
        })
    }
    
    # Submit to control plane
    op_id = await submit_bulk_write(
        destination="VARIANT_TEST_TABLE",
        data=complex_data,
        orm_type="raw"
    )
    
    print(f"✅ Submitted complex nested data: {op_id}")
    print(f"   Event: {complex_data['event_name']}")
    print(f"   Nested levels: ~5-6 deep")
    
    return op_id

# Run example
complex_op = await test_complex_variant_writes()

## 6. Example 3: Batch Operations from DataFrame

In [ ]:
async def test_dataframe_variant_writes():
    """Submit bulk writes from pandas DataFrame"""
    print("📝 Example 3: Batch Operations from DataFrame\n")
    
    # Create sample DataFrame
    df = pd.DataFrame([
        {
            'event_name': 'purchase',
            'user_id': 3001,
            'status': 'completed',
            'amount': 199.99,
            'items': ['item1', 'item2', 'item3'],
            'shipping_address': {
                'street': '123 Main St',
                'city': 'Seattle',
                'state': 'WA',
                'zip': '98101'
            }
        },
        {
            'event_name': 'purchase',
            'user_id': 3002,
            'status': 'completed',
            'amount': 49.99,
            'items': ['item4'],
            'shipping_address': {
                'street': '456 Oak Ave',
                'city': 'Portland',
                'state': 'OR',
                'zip': '97201'
            }
        },
        {
            'event_name': 'purchase',
            'user_id': 3003,
            'status': 'pending',
            'amount': 299.99,
            'items': ['item5', 'item6'],
            'shipping_address': {
                'street': '789 Pine Rd',
                'city': 'San Francisco',
                'state': 'CA',
                'zip': '94102'
            }
        }
    ])
    
    print("📊 DataFrame to insert:")
    print(df[['event_name', 'user_id', 'amount', 'status']])
    print(f"\n{len(df)} rows to process\n")
    
    # Convert DataFrame rows to control plane writes
    operation_ids = []
    for idx, row in df.iterrows():
        # Prepare data with VARIANT serialization
        data = {
            'event_name': row['event_name'],
            'user_id': int(row['user_id']),
            'status': row['status'],
            'event_data': json.dumps({
                'amount': float(row['amount']),
                'currency': 'USD',
                'payment_method': 'credit_card'
            }),
            'user_metadata': json.dumps({
                'items_purchased': row['items'],
                'item_count': len(row['items'])
            }),
            'raw_payload': json.dumps({
                'shipping_address': row['shipping_address'],
                'order_timestamp': datetime.now().isoformat()
            })
        }
        
        op_id = await submit_bulk_write(
            destination="VARIANT_TEST_TABLE",
            data=data,
            orm_type="pandas"
        )
        operation_ids.append(op_id)
        print(f"✅ Submitted row {idx + 1}: {op_id}")
    
    print(f"\n✅ Total operations submitted: {len(operation_ids)}")
    return operation_ids

# Run example
df_ops = await test_dataframe_variant_writes()

## 7. Example 4: ORM-Specific Batching

In [ ]:
async def test_orm_specific_batching():
    """Demonstrate ORM-agnostic batching"""
    print("📝 Example 4: ORM-Specific Batching\n")
    
    # Same destination, different ORMs = separate batches
    orms = ['sqlalchemy', 'django', 'peewee', 'raw']
    
    operation_ids = {}
    
    for orm in orms:
        print(f"\n📝 Submitting {orm.upper()} operations:")
        ops = []
        
        for i in range(3):
            data = {
                'event_name': f'{orm}_event',
                'user_id': 4000 + i,
                'status': 'success',
                'event_data': json.dumps({
                    'orm_type': orm,
                    'record_number': i,
                    'timestamp': datetime.now().isoformat()
                }),
                'user_metadata': json.dumps({
                    'orm_version': '1.0.0',
                    'serialization': 'json'
                })
            }
            
            op_id = await submit_bulk_write(
                destination="VARIANT_TEST_TABLE",
                data=data,
                orm_type=orm  # This groups operations by ORM!
            )
            ops.append(op_id)
            print(f"   ✅ {op_id}")
        
        operation_ids[orm] = ops
    
    # Check stats - should see separate buffers
    print("\n📊 Checking buffer grouping...")
    stats = await get_control_plane_stats()
    print_stats(stats)
    
    print("\n💡 Each ORM type creates a separate buffer group!")
    print("   This ensures proper serialization per ORM.")
    
    return operation_ids

# Run example
orm_ops = await test_orm_specific_batching()

## 8. Example 5: High-Volume Batch

In [ ]:
async def test_high_volume_writes():
    """Submit high volume of operations to test scaling"""
    print("📝 Example 5: High-Volume Batch Write\n")
    
    num_records = 100
    print(f"Submitting {num_records} operations...\n")
    
    start_time = time.time()
    operation_ids = []
    
    # Submit in batches to avoid overwhelming the API
    batch_size = 20
    for batch_num in range(0, num_records, batch_size):
        batch_ops = []
        
        for i in range(batch_num, min(batch_num + batch_size, num_records)):
            data = {
                'event_name': 'bulk_event',
                'user_id': 5000 + i,
                'status': 'success',
                'event_data': json.dumps({
                    'batch_number': batch_num // batch_size,
                    'record_number': i,
                    'data': f'payload_{i}',
                    'metrics': {
                        'value1': i * 1.5,
                        'value2': i * 2.0,
                        'value3': i * 3.0
                    }
                }),
                'user_metadata': json.dumps({
                    'sequence': i,
                    'timestamp': datetime.now().isoformat()
                })
            }
            
            op_id = await submit_bulk_write(
                destination="VARIANT_TEST_TABLE",
                data=data,
                orm_type="raw"
            )
            batch_ops.append(op_id)
        
        operation_ids.extend(batch_ops)
        print(f"✅ Batch {batch_num // batch_size + 1}: Submitted {len(batch_ops)} operations")
    
    elapsed = time.time() - start_time
    
    print(f"\n✅ Total operations submitted: {len(operation_ids)}")
    print(f"⏱️  Time taken: {elapsed:.2f} seconds")
    print(f"📈 Throughput: {len(operation_ids) / elapsed:.1f} ops/second")
    
    # Check stats to see scaling
    print("\n📊 Checking writer pool scaling...")
    stats = await get_control_plane_stats()
    print_stats(stats)
    
    return operation_ids

# Run example
bulk_ops = await test_high_volume_writes()

## 9. Monitor and Flush

In [ ]:
async def monitor_and_flush():
    """Monitor buffer and trigger manual flush"""
    print("📊 Current Control Plane Status:\n")
    
    # Get current stats
    stats = await get_control_plane_stats()
    print_stats(stats)
    
    buffered = stats['buffer_stats']['total_buffered']
    
    if buffered > 0:
        print(f"\n🔄 Triggering manual flush for {buffered} operations...")
        
        flush_result = await trigger_flush()
        
        print(f"\n✅ Flush complete:")
        print(f"   Batches flushed: {flush_result['batches_flushed']}")
        print(f"   Total operations: {flush_result['total_operations']}")
        
        # Wait a moment for processing
        print("\n⏳ Waiting 3 seconds for processing...")
        await asyncio.sleep(3)
        
        # Check stats again
        print("\n📊 Post-flush status:")
        stats_after = await get_control_plane_stats()
        print_stats(stats_after)
    else:
        print("\n💡 No operations buffered. They may have auto-flushed.")
        print("   Check queue_depth to see if they're being processed.")

# Run monitoring and flush
await monitor_and_flush()

## 10. Wait for Auto-Flush

In [ ]:
async def wait_for_auto_flush(timeout: int = 30):
    """Wait for automatic flush to occur"""
    print(f"⏳ Waiting for automatic flush (up to {timeout}s)...\n")
    
    start_time = time.time()
    last_buffered = None
    
    while time.time() - start_time < timeout:
        stats = await get_control_plane_stats()
        buffered = stats['buffer_stats']['total_buffered']
        last_flush_ago = stats['buffer_stats']['last_flush_ago']
        
        if last_buffered is not None and buffered < last_buffered:
            print(f"\n✅ Flush detected!")
            print(f"   Before: {last_buffered} operations")
            print(f"   After: {buffered} operations")
            print_stats(stats)
            return True
        
        last_buffered = buffered
        elapsed = time.time() - start_time
        
        print(f"\r⏱️  {elapsed:.1f}s | Buffered: {buffered} | Last flush: {last_flush_ago:.1f}s ago", end="")
        await asyncio.sleep(1)
    
    print(f"\n\n⏰ Timeout reached ({timeout}s)")
    print("   You can manually trigger flush with trigger_flush()")
    return False

# Wait for auto-flush
# await wait_for_auto_flush(30)

## 11. Troubleshooting VARIANT Issues

In [ ]:
async def test_variant_edge_cases():
    """Test edge cases with VARIANT columns"""
    print("🔍 Testing VARIANT Edge Cases\n")
    
    # Test 1: NULL values
    print("1️⃣ Test: NULL VARIANT values")
    data_with_nulls = {
        'event_name': 'null_test',
        'user_id': 9001,
        'status': 'success',
        'event_data': json.dumps({'key': None}),  # JSON null
        'user_metadata': None,  # Python None = SQL NULL
        'raw_payload': json.dumps(None)  # JSON null as string
    }
    op1 = await submit_bulk_write("VARIANT_TEST_TABLE", data_with_nulls)
    print(f"   ✅ Submitted: {op1}\n")
    
    # Test 2: Empty objects
    print("2️⃣ Test: Empty objects")
    data_empty = {
        'event_name': 'empty_test',
        'user_id': 9002,
        'status': 'success',
        'event_data': json.dumps({}),  # Empty object
        'user_metadata': json.dumps([]),  # Empty array
        'raw_payload': json.dumps('')  # Empty string
    }
    op2 = await submit_bulk_write("VARIANT_TEST_TABLE", data_empty)
    print(f"   ✅ Submitted: {op2}\n")
    
    # Test 3: Special characters
    print("3️⃣ Test: Special characters in VARIANT")
    data_special = {
        'event_name': 'special_chars_test',
        'user_id': 9003,
        'status': 'success',
        'event_data': json.dumps({
            'unicode': '🎉 Hello 世界 مرحبا',
            'quotes': 'He said "Hello"',
            'newlines': 'Line1\nLine2\nLine3',
            'special': '<>&"\'\''
        }),
        'user_metadata': json.dumps({
            'escaped': r'\n\t\r',
            'path': 'C:\\Users\\Documents'
        })
    }
    op3 = await submit_bulk_write("VARIANT_TEST_TABLE", data_special)
    print(f"   ✅ Submitted: {op3}\n")
    
    # Test 4: Large VARIANT data
    print("4️⃣ Test: Large VARIANT object")
    large_object = {
        f'key_{i}': f'value_{i}' * 100 for i in range(100)
    }
    data_large = {
        'event_name': 'large_variant_test',
        'user_id': 9004,
        'status': 'success',
        'event_data': json.dumps(large_object),
        'user_metadata': json.dumps({'size_kb': len(json.dumps(large_object)) / 1024})
    }
    op4 = await submit_bulk_write("VARIANT_TEST_TABLE", data_large)
    print(f"   ✅ Submitted: {op4}")
    print(f"   Size: {len(json.dumps(large_object)) / 1024:.2f} KB\n")
    
    print("✅ All edge cases submitted successfully!")
    
    return [op1, op2, op3, op4]

# Test edge cases
edge_case_ops = await test_variant_edge_cases()

## 12. Summary and Best Practices

### ✅ Key Takeaways:

1. **Always serialize VARIANT data**
   ```python
   'event_data': json.dumps({'key': 'value'})
   ```

2. **Use appropriate ORM types for batching**
   - Same destination + same ORM = single batch
   - Different ORMs = separate batches

3. **Handle NULL properly**
   - Python `None` → SQL `NULL`
   - `json.dumps(None)` → JSON `"null"` string

4. **Monitor buffer status**
   - Check stats to see buffered operations
   - Manual flush for immediate writes
   - Auto-flush happens every 20s (default)

5. **Benefits of Control Plane**
   - Automatic batching (15-30s pooling)
   - Dynamic scaling (1-8 writers)
   - No connection management needed
   - Resilient (SQLite backup)

### 🚀 Performance Tips:

- Submit operations in batches (don't wait for responses)
- Use high priority for time-sensitive data
- Monitor queue depth for scaling insights
- Let auto-flush handle most cases
- Use manual flush only when needed

### 📊 Monitoring:

```python
stats = await get_control_plane_stats()
print(f"Buffered: {stats['buffer_stats']['total_buffered']}")
print(f"Workers: {stats['writer_pool']['active_workers']}")
print(f"Queue: {stats['writer_pool']['queue_depth']}")
```

## 13. Final Summary

Run this cell to see a summary of all operations submitted in this notebook.

In [ ]:
async def final_summary():
    """Display final summary of all operations"""
    print("\n" + "="*60)
    print("📊 FINAL SUMMARY")
    print("="*60)
    
    stats = await get_control_plane_stats()
    
    storage = stats['storage_stats']
    print(f"\n💾 Total Operations:")
    print(f"   Pending: {storage['pending_operations']}")
    print(f"   Completed: {storage['completed_operations']}")
    print(f"   Total: {storage['pending_operations'] + storage['completed_operations']}")
    
    if storage['destination_counts']:
        print(f"\n📍 By Destination:")
        for dest, count in storage['destination_counts'].items():
            print(f"   {dest}: {count} operations")
    
    buffer = stats['buffer_stats']
    print(f"\n📦 Current Buffer:")
    print(f"   Buffered: {buffer['total_buffered']}")
    print(f"   Last Flush: {buffer['last_flush_ago']:.1f}s ago")
    
    writer_pool = stats['writer_pool']
    print(f"\n⚙️  Writer Pool:")
    print(f"   Active Workers: {writer_pool['active_workers']}")
    print(f"   Queue Depth: {writer_pool['queue_depth']}")
    
    print(f"\n✅ All operations submitted successfully!")
    print(f"   Operations will be written to Snowflake in batches.")
    print(f"   Monitor progress at: {FLEETQ_URL}/control-plane/stats")
    print("="*60)

await final_summary()